# Phase 6e Wave 3: Nonlinear Diffeomorphism Invariance — Technical Notebook

Companion to `papers/paper41_diff_invariance/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/NonlinearDiffInvariance.lean` (10 substantive theorems, 0 sorry, 0 new axioms; verified `propext, Classical.choice, Quot.sound` only via `lean_verify`).

**Structure (mirrors paper §3–§5):**
1. Setup — Wave 1 + Wave 2 inputs
2. EffectiveLagrangianCoefs bundle + Dirac instance
3. Path-(b) anomaly residual at orders a₀, a₂, a₄
4. Order-by-order DiffInvariantAt witnesses
5. Correctness-push biconditional
6. Falsifier with linear residual = δ
7. Anomaly hunt over parameter grid
8. Figure: order-by-order check
9. Lean theorem inventory

All numerical content imports from `src.diff_invariance` — no inline physics redefinition.


## 1. Setup — Wave 1 + Wave 2 inputs

In [1]:
import numpy as np
from src.core.constants import DIFF_INVARIANCE_PARAMS

print(f"Path-(b) tolerance     = {DIFF_INVARIANCE_PARAMS['PATH_B_RESIDUAL_TOLERANCE']:.0e}")
print(f"Order list             = {DIFF_INVARIANCE_PARAMS['ORDER_LIST']}")
print(f"Test grid R²    range = {DIFF_INVARIANCE_PARAMS['TEST_GRID_RICCI_SQ_RANGE']}")
print(f"Test grid Ricci² range = {DIFF_INVARIANCE_PARAMS['TEST_GRID_RICCI_SQ_RANGE']}")
print(f"Test grid Riem²  range = {DIFF_INVARIANCE_PARAMS['TEST_GRID_RIEMANN_SQ_RANGE']}")
print(f"Anomaly probe offset   = {DIFF_INVARIANCE_PARAMS['ANOMALY_PROBE_OFFSET']:.0e}")
print(f"Parameter scan points  = {DIFF_INVARIANCE_PARAMS['PARAMETER_SCAN_POINTS']}")

Path-(b) tolerance     = 1e-12
Order list             = (0, 2, 4)
Test grid R²    range = (0.0, 50.0)
Test grid Ricci² range = (0.0, 50.0)
Test grid Riem²  range = (0.0, 25.0)
Anomaly probe offset   = 1e-06
Parameter scan points  = 16


## 2. Effective Lagrangian coefficient bundle + Dirac instance

The Wave 3 abstraction is a 5-tuple $L = (a_0,\, a_{2,R},\, a_{4,R^2},\, a_{4,\mathrm{Ric}^2},\, a_{4,\mathrm{Riem}^2})$.
The canonical Wave 1 Christensen-Duff Dirac bundle invokes each Wave 1 coefficient by name (P6 cross-module bridge integrity).

In [2]:
from src.diff_invariance import dirac_coefficient_bundle, perturbed_coefficient_bundle

N_f = 24.0
b = dirac_coefficient_bundle(N_f)
print(f"diracCoefBundle(N_f={N_f}):")
print(f"  a0            = {b.a0:+.6e}")
print(f"  a2_R          = {b.a2_R:+.6e}")
print(f"  a4_R_sq       = {b.a4_R_sq:+.6e}")
print(f"  a4_Ricci_sq   = {b.a4_Ricci_sq:+.6e}")
print(f"  a4_Riemann_sq = {b.a4_Riemann_sq:+.6e}")
print()

# Perturbed bundle for the falsifier
delta = 1e-3
pb = perturbed_coefficient_bundle(N_f, delta)
print(f"perturbedCoefBundle(N_f={N_f}, δ={delta}):")
print(f"  a4_R_sq (shifted) = {pb.a4_R_sq:+.6e}  (vs Dirac {b.a4_R_sq:+.6e}, Δ = +{delta})")

diracCoefBundle(N_f=24.0):
  a0            = +6.079271e-01
  a2_R          = -1.266515e-02
  a4_R_sq       = -3.518097e-04
  a4_Ricci_sq   = +4.925335e-04
  a4_Riemann_sq = -8.443432e-04

perturbedCoefBundle(N_f=24.0, δ=0.001):
  a4_R_sq (shifted) = +6.481903e-04  (vs Dirac -3.518097e-04, Δ = +0.001)


## 3. Path-(b) anomaly residual at orders a₀, a₂, a₄

Order $a_0$ — constant scalar, no curvature dependence — residual is definitionally zero.
Order $a_2$ — only one scalar invariant ($R$ itself) — no basis change exists — residual is definitionally zero.
Order $a_4$ — load-bearing case — residual is the Wave 2 basis-change difference.

In [3]:
from src.diff_invariance import pathB_residual_at_order

R_test    = 1.5
R_sq_test = 100.0
Ric_test  = 50.0
Riem_test = 25.0

print("Order-a₀ residual (Dirac):    ",
      pathB_residual_at_order(0, b, N_f, R_test, R_sq_test, Ric_test, Riem_test))
print("Order-a₂ residual (Dirac):    ",
      pathB_residual_at_order(2, b, N_f, R_test, R_sq_test, Ric_test, Riem_test))
print("Order-a₄ residual (Dirac):    ",
      pathB_residual_at_order(4, b, N_f, R_test, R_sq_test, Ric_test, Riem_test))
print()
print("Order-a₄ residual (perturbed):",
      pathB_residual_at_order(4, pb, N_f, R_test, R_sq_test, Ric_test, Riem_test))

Order-a₀ residual (Dirac):     0.0
Order-a₂ residual (Dirac):     0.0
Order-a₄ residual (Dirac):     -6.938893903907228e-18

Order-a₄ residual (perturbed): 0.09999999999999999


## 4. Order-by-order `DiffInvariantAt` witnesses

In [4]:
from src.diff_invariance import diff_invariant_at_order, diff_invariant_order_by_order

for order in DIFF_INVARIANCE_PARAMS['ORDER_LIST']:
    ok = diff_invariant_at_order(order, b, N_f, R_test, R_sq_test, Ric_test, Riem_test)
    print(f"DiffInvariantAt(Dirac, N_f={N_f:.0f}, n={order}): {ok}")

print()
print("H_NonlinearDiffInvariance(Dirac):    ",
      diff_invariant_order_by_order(b, N_f, R_test, R_sq_test, Ric_test, Riem_test))
print("H_NonlinearDiffInvariance(perturbed):",
      diff_invariant_order_by_order(pb, N_f, R_test, R_sq_test, Ric_test, Riem_test))

DiffInvariantAt(Dirac, N_f=24, n=0): True
DiffInvariantAt(Dirac, N_f=24, n=2): True
DiffInvariantAt(Dirac, N_f=24, n=4): True

H_NonlinearDiffInvariance(Dirac):     True
H_NonlinearDiffInvariance(perturbed): False


## 5. Correctness-push biconditional

**Lean theorem `diff_invariance_a4_iff_dirac_basis_consistent`:**

$$
\text{DiffInvariantAt}(\text{diracCoefBundle}(N_f),\, N_f,\, 4)
\iff
\forall\, R^2, R_{\mu\nu}^2, R_{\mu\nu\rho\sigma}^2,\
\text{a4 density (natural basis)} = \text{a4 density (Stelle basis)}.
$$

The biconditional makes the load-bearing content of Wave 3 explicit: order-$a_4$ path-(b) invariance is
**logically equivalent** to the Wave 2 basis-change identity holding pointwise. Algebraic equivalence,
not implication — either side falsifies the other.

In [5]:
# Numerically verify the biconditional at multiple curvature inputs.
import math
from src.higher_curvature.curvature_basis import (
    a4_density,
    a4_density_in_RC2GB_basis,
)

print(f"{'R²':>10s} {'Ric²':>10s} {'Riem²':>10s} {'natural':>15s} {'Stelle':>15s} {'diff':>15s}")
print("-" * 80)
for R_sq, Ric, Riem in [(1.0, 0.0, 0.0), (100, 50, 25), (1.5, 3.7, 2.1), (1e6, 5e5, 2.5e5)]:
    nat = a4_density(N_f, R_sq, Ric, Riem)
    stl = a4_density_in_RC2GB_basis(N_f, R_sq, Ric, Riem)
    print(f"{R_sq:>10.2e} {Ric:>10.2e} {Riem:>10.2e} {nat:>15.6e} {stl:>15.6e} {nat - stl:>15.2e}")

        R²       Ric²      Riem²         natural          Stelle            diff
--------------------------------------------------------------------------------
  1.00e+00   0.00e+00   0.00e+00   -3.518097e-04   -3.518097e-04        5.42e-20
  1.00e+02   5.00e+01   2.50e+01   -3.166287e-02   -3.166287e-02       -6.94e-18
  1.50e+00   3.70e+00   2.10e+00   -4.784611e-04   -4.784611e-04       -4.34e-19
  1.00e+06   5.00e+05   2.50e+05   -3.166287e+02   -3.166287e+02        5.68e-14


## 6. Falsifier with linear residual = δ at unit R²

**Lean theorem `perturbed_pathB_residual_a4_at_unit_R_sq`:** at $R^2 = 1, R_{\mu\nu}^2 = R_{\mu\nu\rho\sigma}^2 = 0$, the perturbed-bundle residual equals exactly $\delta$.
The falsifier is **non-tautological** — the residual responds linearly to the probe parameter.

In [6]:
# Numerically verify residual = δ over 9 decades of δ
deltas = np.logspace(-9, 0, 10)
residuals = np.array([
    pathB_residual_at_order(4, perturbed_coefficient_bundle(N_f, d), N_f, 0.0, 1.0, 0.0, 0.0)
    for d in deltas
])
print(f"{'δ':>15s} {'residual':>15s} {'rel err':>15s}")
print("-" * 50)
for d, r in zip(deltas, residuals):
    rel_err = abs(r - d) / abs(d) if d != 0 else 0.0
    print(f"{d:>15.3e} {r:>15.3e} {rel_err:>15.3e}")

              δ        residual         rel err
--------------------------------------------------
      1.000e-09       1.000e-09       6.996e-11
      1.000e-08       1.000e-08       4.903e-12
      1.000e-07       1.000e-07       5.665e-13
      1.000e-06       1.000e-06       7.856e-14
      1.000e-05       1.000e-05       7.962e-15
      1.000e-04       1.000e-04       4.066e-16
      1.000e-03       1.000e-03       0.000e+00
      1.000e-02       1.000e-02       0.000e+00
      1.000e-01       1.000e-01       0.000e+00
      1.000e+00       1.000e+00       0.000e+00


## 7. Anomaly hunt over the parameter grid

In [7]:
from src.diff_invariance import (
    anomaly_hunt_dirac_passes,
    anomaly_hunt_perturbed_fails,
    max_residual_over_grid,
)
from src.diff_invariance.anomaly_hunt import report_anomaly_hunt

print(f"Dirac max residual over grid:     {max_residual_over_grid(N_f, 'dirac'):.3e}")
print(f"Perturbed max residual (δ=1e-6):  {max_residual_over_grid(N_f, 'perturbed', 1e-6):.3e}")
print()
print(f"anomaly_hunt_dirac_passes(N_f={N_f}):     {anomaly_hunt_dirac_passes(N_f)}")
print(f"anomaly_hunt_perturbed_fails(N_f={N_f}): {anomaly_hunt_perturbed_fails(N_f)}")
print()

rep = report_anomaly_hunt(N_f)
for k, v in rep.items():
    print(f"  {k:>26s}: {v}")

Dirac max residual over grid:     1.041e-17
Perturbed max residual (δ=1e-6):  5.000e-05

anomaly_hunt_dirac_passes(N_f=24.0):     True
anomaly_hunt_perturbed_fails(N_f=24.0): True

                         N_f: 24.0
                       delta: 1e-06
                   tolerance: 1e-12
          dirac_max_residual: 1.0408340855860843e-17
      perturbed_max_residual: 5.000000000000143e-05
                dirac_passes: True
             perturbed_fails: True


## 8. Figure: order-by-order path-(b) check

In [8]:
from src.core.visualizations import fig_diff_invariance_order_check
fig = fig_diff_invariance_order_check()
fig.show()

## 9. Lean theorem inventory

The 10 substantive theorems of `NonlinearDiffInvariance.lean`:

| # | Theorem | Content |
|---|---------|---------|
| 1 | `diracCoefBundle_density_a4_eq_wave2_a4_density` | Bridge: bundle.density_a4 = Wave 2 a4_density |
| 2 | `pathB_residual_a4_dirac_eq_zero` | **MAIN.** Order-a₄ residual vanishes for the Dirac bundle (consumes Wave 2 basis-change identity) |
| 3 | `dirac_diffInvariantAt_zero` | Dirac satisfies path-(b) at order 0 |
| 4 | `dirac_diffInvariantAt_two`  | Dirac satisfies path-(b) at order 2 |
| 5 | `dirac_diffInvariantAt_four` | Dirac satisfies path-(b) at order 4 |
| 6 | `diff_invariance_a4_iff_dirac_basis_consistent` | **CORRECTNESS-PUSH biconditional** |
| 7 | `perturbed_pathB_residual_a4_at_unit_R_sq` | Falsifier: residual = δ at unit R² |
| 8 | `perturbed_not_diffInvariantAt_four` | Falsifier-witness: nonzero δ ⇒ NOT diff-invariant |
| 9 | `dirac_H_NonlinearDiffInvariance` | Tracked-Prop witness for Dirac bundle |
| 10 | `perturbed_not_H_NonlinearDiffInvariance` | Tracked-Prop falsifier for perturbed bundle |

**Decision Gate E.3 verdict:** PASS through order a₄. Phase 6e Wave 4 (`NonlinearEFE.lean`) and beyond proceed at full scope.
